In [ ]:
# from datetime import timedelta
# import pandas as pd
# import numpy as np

# # Load the outage data
# outage_df = pd.read_csv("data_collection_by_hollis/western_interconnection_outages_fips_filtered.csv")
# outage_df['start_time'] = pd.to_datetime(outage_df['start_time'])
# outage_df['end_time'] = pd.to_datetime(outage_df['end_time'])



In [7]:
from datetime import timedelta
import pandas as pd

outage_df = pd.read_csv("data_collection_by_hollis/western_interconnection_outages_fips_filtered.csv", parse_dates=["start_time", "end_time"])

exclusion_buffer = timedelta(days=2)
negatives_per_event = 2
negative_samples = []

grouped = outage_df.groupby("fips")

# compute exclusion hours for each county
county_exclusion_hours = {}
for fips, group in grouped:
    hours = set()
    for _, row in group.iterrows():
        exclusion_start = row["start_time"] - exclusion_buffer
        exclusion_end = row["end_time"] + exclusion_buffer
        hours.update(pd.date_range(start=exclusion_start, end=exclusion_end, freq="h"))
    county_exclusion_hours[fips] = hours

# Iterate over outage events
for idx, row in outage_df.iterrows():
    if (idx % 500) == 0:
        print(f"{round(idx / len(outage_df) * 100, 2)}% done")

    fips = row["fips"]
    state = row["state"]
    county = row["county"]
    start_time = row["start_time"]
    end_time = row["end_time"]
    utility_type = row["utility_type"]
    pop_density = row["pop_density"]
    rucc_code = row["rucc_code"]
    land_area_sqmi = row["land_area_sqmi"]

    # Limit search to 180-day range
    window_start = max(pd.Timestamp("2014-01-01"), start_time - timedelta(days=180))
    window_end = min(pd.Timestamp("2023-12-31 23:00:00"), end_time + timedelta(days=180))
    candidate_hours = pd.date_range(start=window_start, end=window_end, freq="h")
    candidate_hours_set = set(candidate_hours)

    exclusion_hours = county_exclusion_hours.get(fips, set())

    valid_hours = list(candidate_hours_set - exclusion_hours)

    if len(valid_hours) >= negatives_per_event:
        sampled = pd.Series(valid_hours).sample(n=negatives_per_event, random_state=idx)
        for ts in sampled:
            negative_samples.append({
                "fips": fips,
                "state": state,
                "county": county,
                "start_time": ts,
                "duration": 0.0,
                "min_customers": 0,
                "max_customers": 0,
                "mean_customers": 0,
                "end_time": ts,
                "event_id": f"{state}-NA-{idx}",
                "state_event": state,
                "datetime_event_began": ts,
                "datetime_restoration": ts,
                "event_type": "None (Normal Operation)",
                "year": ts.year,
                "utility_type": utility_type,
                "pop_density": pop_density,
                "rucc_code": rucc_code,
                "land_area_sqmi": land_area_sqmi
            })

# Save
negative_df = pd.DataFrame(negative_samples)
negative_df.to_csv("data_collection_by_hollis/negatives_per_outage.csv", index=False)

print(f"Done! Saved {len(negative_df)} negatives.")

0.0% done
2.87% done
5.74% done
8.61% done
11.48% done
14.35% done
17.22% done
20.09% done
22.96% done
25.83% done
28.7% done
31.57% done
34.45% done
37.32% done
40.19% done
43.06% done
45.93% done
48.8% done
51.67% done
54.54% done
57.41% done
60.28% done
63.15% done
66.02% done
68.89% done
71.76% done
74.63% done
77.5% done
80.37% done
83.24% done
86.11% done
88.98% done
91.85% done
94.72% done
97.59% done
✅ Done! Saved 34838 negatives.
